# 01 — Inspeção das bases processadas para a tabela mestra

**Escopo desta etapa:** conhecer as fontes, seus campos, datas, qualidade e identificação dos ativos. Não construímos o painel diário, o alvo, sequências ou modelo; não alteramos nem exportamos bases.

A futura unidade de análise será **estação + plataforma + porta + dia**, verificando se o tipo de ativo também precisa compor a chave. O alvo pretendido é a ocorrência de falha entre D+1 e D+7. ZM01 é evidência principal; ZM02 e ZM03 descrevem histórico de manutenção. Manutenção programada ou preventiva não equivale automaticamente a falha.

Usamos exclusivamente os quatro arquivos tratados em `data/`. As abas de auditoria são inspecionadas separadamente, sem concatená-las às bases principais.

In [1]:
from pathlib import Path
import re
import sys
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 100)

# Funciona iniciando o Jupyter na raiz ou em uma subpasta do repositório.
candidatos = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p for p in candidatos
             if (p / "data/Corretiva Emergencial/processados/ZM01_limpa.xlsx").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra o notebook a partir do repositório que contém data/.")

FONTES = {
    "Alarmes": (RAIZ / "data/Alarmes/alarmes_trata_limpos_consolidado.csv", None),
    "ZM01": (RAIZ / "data/Corretiva Emergencial/processados/ZM01_limpa.xlsx", "ZM01_limpa"),
    "ZM02": (RAIZ / "data/Corretiva Programada/processados/ZM02_limpa.xlsx", "ZM02_limpa"),
    "ZM03": (RAIZ / "data/Preventiva/preventiva_zm03_limpa.csv", None),
}
for nome, (arquivo, _) in FONTES.items():
    if not arquivo.is_file():
        raise FileNotFoundError(f"{nome}: {arquivo}")
print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__}")
print(f"Raiz: {RAIZ}")


Python: 3.14.7 | pandas: 3.0.5
Raiz: c:\Users\Inteli\Downloads\EC-M7\g04


## 1. Inventário e leitura

Os CSVs são lidos em UTF-8. As abas são escolhidas explicitamente para não confundir a base limpa com auditorias ou subconjuntos. A contagem de linhas abaixo corresponde aos registros lidos, não à quantidade de episódios de falha.

In [2]:
bases = {}
auditorias = {}
inventario = []
for nome, (arquivo, principal) in FONTES.items():
    if principal is None:
        df = pd.read_csv(arquivo, encoding="utf-8-sig", low_memory=False)
        bases[nome] = df
        inventario.append({"fonte": nome, "arquivo": str(arquivo.relative_to(RAIZ)),
                           "aba": "(CSV)", "papel": "principal",
                           "linhas": len(df), "colunas": len(df.columns)})
    else:
        with pd.ExcelFile(arquivo) as livro:
            for aba in livro.sheet_names:
                df = pd.read_excel(livro, sheet_name=aba)
                if aba == principal:
                    bases[nome] = df
                else:
                    auditorias[(nome, aba)] = df
                inventario.append({"fonte": nome, "arquivo": str(arquivo.relative_to(RAIZ)),
                                   "aba": aba, "papel": "principal" if aba == principal else "auditoria",
                                   "linhas": len(df), "colunas": len(df.columns)})
display(pd.DataFrame(inventario))
for (nome, aba), df in auditorias.items():
    if aba == "Decisoes_limpeza":
        print(f"Decisões de limpeza — {nome}")
        display(df)


,fonte,arquivo,aba,papel,linhas,colunas
0,Alarmes,data\Alarmes\alarmes_trata_limpos_consolidado.csv,(CSV),principal,742506,15
1,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,ZM01_limpa,principal,10543,27
2,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Outliers_removidos,auditoria,103,27
3,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Qualidade_ausencias,auditoria,27,3
4,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Decisoes_limpeza,auditoria,7,2
5,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,ZM02_limpa,principal,1163,27
6,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Duracoes_extremas,auditoria,12,27
7,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Duracoes_zero,auditoria,5,27
8,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Qualidade_ausencias,auditoria,27,3
9,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Decisoes_limpeza,auditoria,9,2


Decisões de limpeza — ZM01


,campo_ou_regra,decisao
0,iqm,Coluna removida por conter 100% de valores ausentes.
1,duplicatas,Removidas 1 linha(s) exatamente duplicada(s).
2,duracao_avaria_horas,"Removidos 103 registros acima do percentil 99 (25.00 horas), preservados separadamente para audi..."
3,fim_da_avaria,Ausências preservadas; não foi realizada imputação.
4,data_analise,Ausências preservadas; marca e data de análise não são equivalentes.
5,campos_catalogo,"Ausências de objeto, defeito, causa e ação foram preservadas."
6,ordens_ausentes,Registros sem ordem foram mantidos quando possuíam nota e dados úteis.


Decisões de limpeza — ZM02


,campo_ou_regra,decisao
0,falha,Removida por conter 100% de valores ausentes.
1,iqm,Removida por conter 100% de valores ausentes.
2,duplicatas,Nenhuma duplicata exata encontrada após a limpeza.
3,ordens_repetidas,Cinco ordens repetidas foram mantidas porque apresentam divergências em campos técnicos.
4,ordens_ausentes,"Cinco registros sem ordem foram mantidos por possuírem nota, descrição e localização."
5,valores_ausentes,As ausências foram preservadas sem imputação.
6,duracao_extrema,Foram sinalizados 12 registros acima do percentil 99 (6104.70 horas). Os registros foram mantido...
7,duracao_zero,"Foram sinalizados 5 registros com duração igual a zero, mantidos sem imputação."
8,colunas_esparsas,As colunas comentarios_classificacao_engenharia e analise_datetime foram removidas da base final...


## 2. Colunas, tipos e ausências

Tipos são os inferidos na leitura. Não fazemos preenchimento de ausências nem remoção de registros. As amostras exibem somente identificação e datas, evitando reproduzir textos longos das ordens.

In [3]:
AMOSTRAS = {
    "Alarmes": ["TAG", "estacao_extraida", "plataforma_extraida", "porta_extraida",
                "tipo_evento_extraido", "dt_alarme", "dt_normalizacao"],
    "ZM01": ["nota", "ordem", "local_de_instalacao", "inicio_avaria_datetime", "fim_avaria_datetime"],
    "ZM02": ["nota", "ordem", "local_de_instalacao", "inicio_avaria_datetime", "fim_avaria_datetime"],
    "ZM03": ["Ordem", "Local de instalação", "DT_Inicio", "DT_Fim", "cancelada"],
}
for nome, df in bases.items():
    print(f"\n{nome}")
    perfil = pd.DataFrame({
        "coluna": df.columns,
        "tipo_lido": df.dtypes.astype(str).values,
        "ausentes": df.isna().sum().values,
        "ausentes_pct": df.isna().mean().mul(100).round(2).values,
        "strings_vazias": [df[c].map(lambda v: isinstance(v, str) and not v.strip()).sum()
                           if df[c].dtype == "object" else 0 for c in df.columns],
    })
    display(perfil)
    display(df[AMOSTRAS[nome]].head(5))



Alarmes


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,TAG,str,0,0.00,0
1,Horário do Alarme,str,0,0.00,0
2,Estação,str,0,0.00,0
3,Porta,float64,170,0.02,0
4,Alarme,str,0,0.00,0
5,Horário da Normalizacao,str,0,0.00,0
6,Tempo de Normalização,str,0,0.00,0
7,tempo_normalizacao_seg,int64,0,0.00,0
8,dt_alarme,str,0,0.00,0
9,dt_normalizacao,str,0,0.00,0


,TAG,estacao_extraida,plataforma_extraida,porta_extraida,tipo_evento_extraido,dt_alarme,dt_normalizacao
0,PSD_FAL_FAL2_PDM09_OBS,FAL,FAL2,9.0,OBS,2026-05-12 08:16:52.657,2026-05-12 08:16:53.670
1,PSD_LUZ_LUZ1_PDM08_OBS,LUZ,LUZ1,8.0,OBS,2026-05-12 08:15:09.913,2026-05-12 08:15:10.917
2,PSD_FAL_FAL1_PDM15_OBS,FAL,FAL1,15.0,OBS,2026-05-12 08:14:06.657,2026-05-12 08:14:07.670
3,PSD_LUZ_LUZ1_PDM19_OBS,LUZ,LUZ1,19.0,OBS,2026-05-12 08:09:31.930,2026-05-12 08:09:32.917
4,PSD_FRE_FRE1_PDM14_OBS,FRE,FRE1,14.0,OBS,2026-05-12 08:05:05.830,2026-05-12 08:05:14.847



ZM01


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,linha_origem,int64,0,0.00,0
1,tipo_de_nota,str,0,0.00,0
2,nota,int64,0,0.00,0
3,ordem,float64,1,0.01,0
4,perfil_do_catalogo,str,0,0.00,0
5,descricao,str,0,0.00,0
6,local_de_instalacao,str,0,0.00,0
7,centro_de_trabalho_responsavel,str,0,0.00,0
8,inicio_avaria_datetime,datetime64[us],0,0.00,0
9,fim_avaria_datetime,datetime64[us],326,3.09,0


...

,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,linha_origem,int64,0,0.00,0
1,tipo_de_nota,str,0,0.00,0
2,nota,int64,0,0.00,0
3,ordem,float64,5,0.43,0
4,perfil_do_catalogo,str,0,0.00,0
5,descricao,str,0,0.00,0
6,local_de_instalacao,str,0,0.00,0
7,centro_de_trabalho_responsavel,str,0,0.00,0
8,inicio_avaria_datetime,datetime64[us],0,0.00,0
9,fim_avaria_datetime,datetime64[us],39,3.35,0


,nota,ordem,local_de_instalacao,inicio_avaria_datetime,fim_avaria_datetime
0,15069252,54081268.0,SP04-6-PSD-BUT,2026-01-25 08:20:00,2026-01-25 09:30:00
1,14647619,53659484.0,SP04-6-PSD-BUT-P1S1,2025-05-30 09:30:00,2025-06-01 04:00:00
2,14682623,53694340.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-06-19 02:00:00,2025-06-19 04:30:00
3,14864269,53874165.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-09-30 09:00:00,2025-09-30 12:00:00
4,13390549,52423535.0,SP04-6-PSD-BUT-PLT1,2023-07-21 14:00:00,2023-07-27 13:50:56



ZM03


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,Ordem,int64,0,0.00,0
1,Tipo de ordem,str,0,0.00,0
2,Centro de trabalho,str,0,0.00,0
3,Local de instalação,str,0,0.00,0
4,Texto breve,str,0,0.00,0
5,Plano de manutenção,int64,0,0.00,0
6,Status usuário,str,0,0.00,0
7,Status do sistema,str,0,0.00,0
8,Data-base do início,str,0,0.00,0
9,Hr.início programada,str,0,0.00,0


,Ordem,Local de instalação,DT_Inicio,DT_Fim,cancelada
0,54164639,SP04-6-PSD-PTA,2026-03-23 21:30:00,2026-03-24 04:30:00,False
1,54164638,SP04-6-PSD-FAL,2026-03-24 21:30:00,2026-03-25 04:30:00,False
2,54152224,SP04-6-PSD-PIN,2026-03-17 21:30:00,2026-03-18 04:30:00,False
3,54152223,SP04-6-PSD-BUT,2026-03-16 21:30:00,2026-03-17 04:30:00,False
4,54143168,SP04-6-PSD-VSO-PLT1,2026-03-30 21:30:00,2026-04-02 04:30:00,False


## 3. Datas e período dos registros

A conversão ocorre em séries auxiliares; valores inválidos são contabilizados. Mínimo e máximo são **datas de eventos registrados**, não confirmação de cobertura contínua. Meses sem registros também não demonstram ausência de falhas.

Em ZM01/ZM02, `inicio_avaria_datetime` é um campo candidato a início da avaria, não prova de execução da manutenção ou de quando a informação ficou disponível. Em ZM03, comparamos `DT_Inicio`/`DT_Fim` com os campos de programação: equivalência não comprova execução.

In [4]:
DATAS = {
    "Alarmes": ["dt_alarme", "dt_normalizacao"],
    "ZM01": ["inicio_avaria_datetime", "fim_avaria_datetime", "analise_datetime", "encerramento_datetime"],
    "ZM02": ["inicio_avaria_datetime", "fim_avaria_datetime", "encerramento_datetime"],
    "ZM03": ["DT_Inicio", "DT_Fim", "Data-base do início", "Data-base do fim"],
}
datas = {}
resumo_datas = []
for nome, colunas in DATAS.items():
    for coluna in colunas:
        original = bases[nome][coluna]
        convertida = pd.to_datetime(original, errors="coerce", format="mixed")
        datas[nome, coluna] = convertida
        resumo_datas.append({
            "fonte": nome, "coluna": coluna, "menor_data": convertida.min(),
            "maior_data": convertida.max(), "ausentes_origem": original.isna().sum(),
            "nao_convertidas": (original.notna() & convertida.isna()).sum(),
        })
display(pd.DataFrame(resumo_datas))

PARES = {"Alarmes": ("dt_alarme", "dt_normalizacao"),
         "ZM01": ("inicio_avaria_datetime", "fim_avaria_datetime"),
         "ZM02": ("inicio_avaria_datetime", "fim_avaria_datetime"),
         "ZM03": ("DT_Inicio", "DT_Fim")}
for nome, (inicio, fim) in PARES.items():
    a, b = datas[nome, inicio], datas[nome, fim]
    print(f"{nome}: fim anterior ao início = {(b < a).sum()}; fim ausente/inválido = {b.isna().sum()}")

mensais = {}
for nome, coluna in {"Alarmes": "dt_alarme", "ZM01": "inicio_avaria_datetime",
                     "ZM02": "inicio_avaria_datetime", "ZM03": "DT_Inicio"}.items():
    serie = datas[nome, coluna].dropna()
    if not serie.empty:
        contagem = serie.dt.to_period("M").value_counts().sort_index()
        mensais[nome] = contagem.reindex(pd.period_range(contagem.index.min(), contagem.index.max(), freq="M"), fill_value=0)
print("Registros por mês (ausente fora do intervalo de eventos de cada fonte):")
display(pd.DataFrame(mensais).sort_index().astype("Int64"))

for derivada, base in [("DT_Inicio", "Data-base do início"), ("DT_Fim", "Data-base do fim")]:
    a, b = datas["ZM03", derivada], datas["ZM03", base]
    validos = a.notna() & b.notna()
    print(f"ZM03: {derivada} e {base}: {validos.sum()} pares válidos; "
          f"{(a[validos].dt.normalize() == b[validos].dt.normalize()).sum()} no mesmo dia.")
display(bases["ZM03"][["Data-base do início", "Hr.início programada", "DT_Inicio",
                       "Data-base do fim", "Fim-base (hora)", "DT_Fim"]].head())


,fonte,coluna,menor_data,maior_data,ausentes_origem,nao_convertidas
0,Alarmes,dt_alarme,2023-02-27 06:26:56.897,2026-05-12 08:16:52.657,0,0
1,Alarmes,dt_normalizacao,2023-02-27 06:27:11.897,2026-05-12 08:16:53.670,0,0
2,ZM01,inicio_avaria_datetime,2023-01-01 02:39:00.000,2026-04-01 22:43:00.000,0,0
3,ZM01,fim_avaria_datetime,2023-01-01 03:39:00.000,2026-04-01 23:25:57.000,326,0
4,ZM01,analise_datetime,2024-11-14 09:59:15.000,2026-05-06 13:39:20.000,5756,0
5,ZM01,encerramento_datetime,2023-01-01 05:16:11.000,2026-04-30 14:01:20.000,1,0
6,ZM02,inicio_avaria_datetime,2023-01-03 03:55:59.000,2026-03-28 14:15:00.000,0,0
7,ZM02,fim_avaria_datetime,2023-01-07 11:30:00.000,2026-05-14 02:00:00.000,39,0
8,ZM02,encerramento_datetime,2023-01-07 17:07:19.000,2026-05-14 04:33:52.000,7,0
9,ZM03,DT_Inicio,2022-12-26 05:30:00.000,2026-03-30 21:30:00.000,0,0


Alarmes: fim anterior ao início = 0; fim ausente/inválido = 0
ZM01: fim anterior ao início = 0; fim ausente/inválido = 326
ZM02: fim anterior ao início = 0; fim ausente/inválido = 39
ZM03: fim anterior ao início = 0; fim ausente/inválido = 0
Registros por mês (ausente fora do intervalo de eventos de cada fonte):


,Alarmes,ZM01,ZM02,ZM03
2022-12,<NA>,<NA>,<NA>,1
2023-01,<NA>,147,51,22
2023-02,50,171,29,16
2023-03,43,178,40,16
2023-04,0,282,31,19
2023-05,0,293,27,18
2023-06,0,292,35,15
2023-07,0,249,18,19
2023-08,0,310,7,12
2023-09,0,199,37,17


ZM03: DT_Inicio e Data-base do início: 569 pares válidos; 569 no mesmo dia.
ZM03: DT_Fim e Data-base do fim: 569 pares válidos; 567 no mesmo dia.


,Data-base do início,Hr.início programada,DT_Inicio,Data-base do fim,Fim-base (hora),DT_Fim
0,2026-03-23,21:30:00,2026-03-23 21:30:00,2026-03-24,04:30:00,2026-03-24 04:30:00
1,2026-03-24,21:30:00,2026-03-24 21:30:00,2026-03-25,04:30:00,2026-03-25 04:30:00
2,2026-03-17,21:30:00,2026-03-17 21:30:00,2026-03-18,04:30:00,2026-03-18 04:30:00
3,2026-03-16,21:30:00,2026-03-16 21:30:00,2026-03-17,04:30:00,2026-03-17 04:30:00
4,2026-03-30,21:30:00,2026-03-30 21:30:00,2026-04-02,04:30:00,2026-04-02 04:30:00


## 4. Identificação dos ativos — extração exploratória

As regras abaixo são deliberadamente conservadoras e criam tabelas auxiliares. Formatos desconhecidos ficam para revisão, sem inferência por texto livre.

- SAP: `SP04-6-PSD-BUT-PLT1-PP05PDM` → BUT / 1 / 05 / PDM.
- Alarme: `PSD_BUT_BUT1_PDM05_OBS` → BUT / 1 / 05 / PDM.
- Registros apenas de estação ou plataforma permanecem nesse nível.
- PDM, PFX e outros tipos não são unificados automaticamente.

A chave completa é provisória e inclui o tipo. Não cruzamos eventos nem distribuímos manutenções coletivas entre portas.

In [5]:
def extrair_ativos(serie, alarme=False):
    texto = serie.astype("string").str.strip().str.upper()
    if alarme:
        padrao = r"^PSD_(?P<estacao>[A-Z]{3})_(?P<estacao_plataforma>[A-Z]{3})(?P<plataforma>\d+)_(?P<tipo_ativo>[A-Z]+)(?P<porta>\d+)_(?P<evento>[A-Z0-9]+)$"
        out = texto.str.extract(padrao)
        # Outra convenção presente na base: PSD_P_BUT1_PDM07_FC.
        alternativo = texto.str.extract(
            r"^PSD_P_(?P<estacao>[A-Z]{3})(?P<plataforma>\d+)_(?P<tipo_ativo>[A-Z]+)(?P<porta>\d+)_(?P<evento>[A-Z0-9]+)$")
        alternativo["estacao_plataforma"] = alternativo["estacao"]
        out = out.combine_first(alternativo)
        # Tags com estações divergentes não recebem chave válida.
        divergente = out.estacao.ne(out.estacao_plataforma).fillna(False)
        out.loc[divergente, ["estacao", "plataforma", "porta", "tipo_ativo"]] = pd.NA
    else:
        out = texto.str.extract(
            r"^SP\d+-\d+-PSD-(?P<estacao>[A-Z]{3})(?:-|$)")
        out["plataforma"] = texto.str.extract(r"-PLT(\d+)(?:-|$)", expand=False)
        porta = texto.str.extract(r"-PP(?P<porta>\d+)(?P<tipo_ativo>[A-Z]+)$")
        out = out.join(porta)
    out = out[["estacao", "plataforma", "porta", "tipo_ativo"]].copy()
    for coluna in ["plataforma", "porta"]:
        out[coluna] = pd.to_numeric(out[coluna], errors="coerce").astype("Int64").astype("string")
    completa = out.notna().all(axis=1)
    # Nível identificável não significa que um subativo é uma porta.
    out["nivel_identificavel"] = "não reconhecido"
    out.loc[out.estacao.notna(), "nivel_identificavel"] = "estação"
    out.loc[out.estacao.notna() & out.plataforma.notna(), "nivel_identificavel"] = "plataforma"
    out.loc[completa, "nivel_identificavel"] = "porta e tipo"
    out["chave_candidata"] = out.estacao + "|" + out.plataforma + "|" + out.porta + "|" + out.tipo_ativo
    out["identificacao_original"] = serie
    return out

COLUNA_ATIVO = {"Alarmes": "TAG", "ZM01": "local_de_instalacao",
                "ZM02": "local_de_instalacao", "ZM03": "Local de instalação"}
ativos = {nome: extrair_ativos(df[COLUNA_ATIVO[nome]], alarme=nome == "Alarmes")
          for nome, df in bases.items()}
for nome, df in ativos.items():
    print(f"\n{nome} — quantidade de registros por nível identificável")
    display(df.nivel_identificavel.value_counts().rename_axis("nível").to_frame("registros"))
    print("Exemplos de identificação (incluindo formatos que exigem revisão):")
    display(df.drop_duplicates("identificacao_original").groupby("nivel_identificavel", sort=False).head(4))
    print("Tipos identificados:")
    display(df.tipo_ativo.value_counts(dropna=False).to_frame("registros"))



Alarmes — quantidade de registros por nível identificável


,registros
nível,
porta e tipo,656603
não reconhecido,85903


Exemplos de identificação (incluindo formatos que exigem revisão):


,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,identificacao_original
0,FAL,2,9,PDM,porta e tipo,FAL|2|9|PDM,PSD_FAL_FAL2_PDM09_OBS
1,LUZ,1,8,PDM,porta e tipo,LUZ|1|8|PDM,PSD_LUZ_LUZ1_PDM08_OBS
2,FAL,1,15,PDM,porta e tipo,FAL|1|15|PDM,PSD_FAL_FAL1_PDM15_OBS
3,LUZ,1,19,PDM,porta e tipo,LUZ|1|19|PDM,PSD_LUZ_LUZ1_PDM19_OBS
104431,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,PSD_P_LUZ1_CBTC_BIM_FAULT_Value
241937,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,PSD_P_MBI1_MDU1STS
241945,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,PSD_P_MBI1_MDU3STS
241946,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,PSD_P_MBI1_MDU6STS


Tipos identificados:


...

,registros
nível,
plataforma,313
estação,173
não reconhecido,83


Exemplos de identificação (incluindo formatos que exigem revisão):


,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,identificacao_original
0,PTA,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-PTA
1,FAL,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-FAL
2,PIN,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-PIN
3,BUT,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-BUT
4,VSO,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-VSO-PLT1
9,MBI,2,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-MBI-PLT2
10,MBI,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-MBI-PLT1
11,PTA,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-PTA-PLT1
260,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,SP04-6-PSD


Tipos identificados:


,registros
tipo_ativo,
<NA>,569


## 5. Compatibilidade das chaves

A interseção abaixo mede apenas a existência da mesma chave nas fontes, **sem associação temporal ou causal**. Uma chave ausente pode decorrer de cobertura, nomenclatura ou falta de eventos. A TAG é comparada aos campos já extraídos dos alarmes para conferir consistência.

In [6]:
a = bases["Alarmes"]
extraido = ativos["Alarmes"]
comparacoes = {
    "estacao": a["estacao_extraida"].astype("string").str.upper().str.strip(),
    "plataforma": a["plataforma_extraida"].astype("string").str.extract(r"(\d+)$", expand=False),
    "porta": pd.to_numeric(a["porta_extraida"], errors="coerce").astype("Int64").astype("string"),
}
for campo, existente in comparacoes.items():
    if campo == "plataforma":
        existente = pd.to_numeric(existente, errors="coerce").astype("Int64").astype("string")
    validos = extraido[campo].notna() & existente.notna()
    print(f"Alarmes — {campo}: {validos.sum()} comparáveis, "
          f"{(extraido.loc[validos, campo] != existente[validos]).sum()} divergências.")

chaves_alarmes = set(extraido.chave_candidata.dropna())
resumo_chaves = []
for nome, df in ativos.items():
    chaves = set(df.chave_candidata.dropna())
    resumo_chaves.append({"fonte": nome, "chaves_completas_distintas": len(chaves),
                          "presentes_nos_alarmes": len(chaves & chaves_alarmes),
                          "ausentes_nos_alarmes": len(chaves - chaves_alarmes)})
display(pd.DataFrame(resumo_chaves))

todos = pd.concat([df[["estacao", "plataforma", "porta", "tipo_ativo"]].dropna().drop_duplicates()
                   for df in ativos.values()]).drop_duplicates()
colisoes = todos.groupby(["estacao", "plataforma", "porta"]).tipo_ativo.nunique()
print("Combinações estação/plataforma/número com mais de um tipo:", colisoes.gt(1).sum())
display(colisoes[colisoes.gt(1)].rename("quantidade_tipos").head(10).to_frame())


Alarmes — estacao: 656603 comparáveis, 0 divergências.
Alarmes — plataforma: 656603 comparáveis, 0 divergências.
Alarmes — porta: 656603 comparáveis, 0 divergências.


,fonte,chaves_completas_distintas,presentes_nos_alarmes,ausentes_nos_alarmes
0,Alarmes,886,886,0
1,ZM01,589,474,115
2,ZM02,518,305,213
3,ZM03,0,0,0


Combinações estação/plataforma/número com mais de um tipo: 459


quantidade_tipos
estacao plataforma porta                  
BUT     1          1                     2
                   10                    2
                   11                    2
                   12                    2
                   13                    2
                   15                    3
                   16                    2
                   17                    2
                   18                    3
                   19                    2

## 6. Duplicidades e flags

Repetição de ordem ou nota não implica automaticamente duplicata: pode representar atualização ou vários registros legítimos. Contamos todas as linhas envolvidas (`keep=False`) e também o excedente de duplicatas exatas. Não removemos registros.

In [7]:
resumo_duplicatas = []
for nome, df in bases.items():
    resumo_duplicatas.append({"fonte": nome,
                              "linhas_em_duplicatas_exatas": df.duplicated(keep=False).sum(),
                              "excedente_duplicatas_exatas": df.duplicated().sum()})
display(pd.DataFrame(resumo_duplicatas))
for nome, colunas in {"ZM01": ["ordem", "nota"], "ZM02": ["ordem", "nota"], "ZM03": ["Ordem"]}.items():
    for coluna in colunas:
        serie = bases[nome][coluna].dropna()
        print(f"{nome}/{coluna}: {serie.duplicated(keep=False).sum()} linhas com identificador repetido; "
              f"{serie[serie.duplicated(keep=False)].nunique()} identificadores repetidos.")

chave_evento = ["TAG", "dt_alarme"]
validos = a[chave_evento].notna().all(axis=1)
print("Alarmes: linhas com TAG + dt_alarme repetidos:",
      a.loc[validos].duplicated(chave_evento, keep=False).sum())
for nome, df in bases.items():
    flags = [c for c in df.columns if c.endswith("_flag") or c == "cancelada"]
    for coluna in flags:
        print(f"{nome} — {coluna}")
        display(df[coluna].value_counts(dropna=False).rename_axis("valor").to_frame("registros"))

# Apenas diagnóstico: não reinserir registros excluídos automaticamente.
excluidos = auditorias.get(("ZM01", "Outliers_removidos"))
if excluidos is not None:
    print(f"ZM01: {len(excluidos)} registros preservados em Outliers_removidos.")
    print("Antes do alvo, revisar se são falhas válidas: excluir por duração pode retirar falhas relevantes.")


,fonte,linhas_em_duplicatas_exatas,excedente_duplicatas_exatas
0,Alarmes,0,0
1,ZM01,0,0
2,ZM02,0,0
3,ZM03,0,0


ZM01/ordem: 136 linhas com identificador repetido; 66 identificadores repetidos.
ZM01/nota: 213 linhas com identificador repetido; 104 identificadores repetidos.
ZM02/ordem: 10 linhas com identificador repetido; 5 identificadores repetidos.
ZM02/nota: 10 linhas com identificador repetido; 5 identificadores repetidos.
ZM03/Ordem: 0 linhas com identificador repetido; 0 identificadores repetidos.
Alarmes: linhas com TAG + dt_alarme repetidos: 449829
ZM01 — indisponivel_flag


,registros
valor,
True,6972
False,3571


ZM01 — falha_flag


,registros
valor,
False,7150
True,3393


...

ZM02 — duracao_zero_flag


,registros
valor,
False,1158
True,5


ZM03 — cancelada


,registros
valor,
False,559
True,10


ZM01: 103 registros preservados em Outliers_removidos.
Antes do alvo, revisar se são falhas válidas: excluir por duração pode retirar falhas relevantes.


## 7. Decisões necessárias antes da tabela mestra

1. **Ativos:** validar os formatos não reconhecidos, os tipos elegíveis (por exemplo, PDM) e se tipo deve integrar a chave. Não misturar portas homônimas de plataformas diferentes.
2. **Manutenções coletivas:** definir como históricos de estação/plataforma entram como contexto, sem convertê-los em intervenções individuais comprovadas.
3. **Semântica temporal:** confirmar a data efetiva de cada manutenção e quando o registro ficou disponível. Em especial, não tratar datas programadas da ZM03 como prova de execução. Status atuais e textos atualizados depois do evento não reconstituem automaticamente o estado histórico.
4. **Falha:** definir critérios para episódios ZM01 e eventual evidência de defeitos em ZM02, revisar os registros excluídos na limpeza ZM01 e depois estabelecer a regra de associação com alarmes. ZM03 não é alvo positivo por si só.
5. **Cobertura:** confirmar os intervalos completos de cada fonte e do cadastro de ativos; não deduzir início de operação da primeira ocorrência nem cobertura pelo último evento.
6. **Duplicidades:** revisar ordens/notas repetidas antes de contar intervenções ou episódios.

**Resultado desta etapa:** inventário, perfis de qualidade, períodos de eventos e chaves candidatas em memória. Nenhum arquivo de dados é gerado. A construção do calendário diário e do alvo fica para a próxima etapa, após revisar estes resultados.